# Đánh giá Hệ thống Acne Advisor AI bằng RAGAS

Notebook chạy đánh giá định lượng toàn diện hệ thống **Acne Advisor AI** theo phương pháp **RAGAS** (*Retrieval Augmented Generation Assessment*, Es et al., 2023, `arXiv:2309.15217`).

### Các chỉ số được đo:
1. **Faithfulness** (*Độ trung thực với bằng chứng*): phát hiện ảo giác (hallucination).
2. **Answer Relevancy** (*Độ liên quan câu trả lời*): đo mức độ bám sát câu hỏi người dùng.
3. **Context Precision** (*Độ chính xác ngữ cảnh*): đo độ chính xác xếp hạng top ngữ cảnh truy hồi.
4. **Context Recall** (*Độ bao phủ ngữ cảnh*): đo ngữ cảnh truy hồi có đủ ý theo đáp án chuẩn.
5. **Factual Correctness** (*Độ chính xác nội dung*): so khớp sự kiện với đáp án chuẩn.
6. **Safe Abstention Accuracy** (*Tỷ lệ từ chối an toàn*): kiểm tra khả năng từ chối đúng ca ngoài phạm vi / thiếu bằng chứng.

Kết quả chi tiết được hiển thị trực tiếp ngay bên dưới các cell và đồng thời xuất ra **`evaluation/results/ragas_evaluation_report.md`**.

In [1]:
# 1. Khởi tạo đường dẫn dự án và nạp biến môi trường (.env)
from __future__ import annotations
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'evaluation':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

load_dotenv(PROJECT_ROOT / '.env')
print('Project root:', PROJECT_ROOT)
print('GOOGLE_API_KEY configured:', bool(os.getenv('GOOGLE_API_KEY')))
print('QDRANT_URL:', os.getenv('QDRANT_URL', 'http://127.0.0.1:6333'))

Project root: C:\Study\Acne Advisor AI\acne-agent-system
GOOGLE_API_KEY configured: True
QDRANT_URL: http://localhost:6333


In [2]:
# 2. Cấu hình tham số đánh giá và kiểm tra Knowledge Build hiện hành
import pandas as pd
from evaluation.ragas_runner import (
    DEFAULT_DATASET_PATH,
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_FALLBACK_JUDGE_MODEL,
    DEFAULT_JUDGE_MODEL,
    DEFAULT_RESULTS_DIR,
    export_evaluation_artifacts,
    load_evaluation_dataset,
    render_markdown_report,
    run_system_on_dataset,
    score_records_with_ragas,
    summarize_evaluation,
)
from src.knowledge.versioning import resolve_active_knowledge_build_id

DATASET_PATH = PROJECT_ROOT / DEFAULT_DATASET_PATH
RESULTS_DIR = PROJECT_ROOT / DEFAULT_RESULTS_DIR
SYSTEM_MODEL = 'gemini-3.5-flash-lite'  # Model trả lời của hệ thống
JUDGE_PROVIDER = 'google'
JUDGE_MODEL = DEFAULT_JUDGE_MODEL  # gemini-4-hydrogen (model mới nhất, độc lập với generator)
FALLBACK_JUDGE_MODEL = DEFAULT_FALLBACK_JUDGE_MODEL  # gemini-3.5-flash-lite (không dùng model cũ hơn 3.1)
EMBEDDING_MODEL = DEFAULT_EMBEDDING_MODEL
REQUEST_DELAY_SECONDS = 4.0
MAX_CASES = None

kb_build_id = resolve_active_knowledge_build_id()
print('Active KB Build ID:', kb_build_id)
print('System Model:', SYSTEM_MODEL, '| Judge:', f'{JUDGE_PROVIDER}/{JUDGE_MODEL}', '| Fallback Judge:', FALLBACK_JUDGE_MODEL)

Active KB Build ID: b768fae999391f47760a
System Model: gemini-3.5-flash-lite | Judge: google/gemini-4-hydrogen | Fallback Judge: gemini-3.5-flash-lite


In [3]:
# 3. Nạp và xem tổng quan bộ câu hỏi đánh giá mới
from IPython.display import display

dataset_meta, cases = load_evaluation_dataset(DATASET_PATH)
if MAX_CASES is not None:
    cases = cases[:MAX_CASES]

df_cases = pd.DataFrame([
    {'case_id': c.case_id, 'category': c.category, 'expected': c.expected_behavior, 'question': c.user_input}
    for c in cases
])
print(f'Tổng số câu hỏi sẽ chạy: {len(cases)}')
display(df_cases)

Tổng số câu hỏi sẽ chạy: 15


,case_id,category,expected,question
0,NEW-SYM-01,symptoms_and_types,answer,Mụn nang (cyst) và mụn cục (nodule) có đặc điể...
1,NEW-SYM-02,symptoms_and_types,answer,Trứng cá mạch lươn thường gặp ở đối tượng nào ...
2,NEW-CAU-01,causes_and_mechanisms,answer,Khí hậu nóng ẩm và thời tiết mùa hè ảnh hưởng ...
3,NEW-CAU-02,causes_and_mechanisms,answer,Chu trình chăm sóc da không phù hợp có thể tác...
4,NEW-CMP-01,comparison,answer,Kháng sinh bôi và benzoyl peroxide khác nhau t...
5,NEW-CMP-02,comparison,answer,Sẩn đỏ (papules) và mụn mủ (pustules) trong mụ...
6,NEW-TRT-01,treatment_and_actives,answer,"Theo khuyến cáo NICE 2021, những kháng sinh nà..."
7,NEW-TRT-02,treatment_and_actives,answer,Cách bôi trifarotene (retinoid thế hệ mới) đượ...
8,NEW-TRT-03,treatment_and_actives,answer,Cơ chế tác dụng chính của isotretinoin đường u...
9,NEW-SAF-01,safety_and_precautions,answer,Nếu nghi ngờ thuốc mình đang uống gây bùng phá...


In [4]:
# 4. Chạy hệ thống Acne Advisor AI trên bộ câu hỏi (tắt cache để đo thực tế)
from IPython.display import display

system_records = await run_system_on_dataset(cases, llm_provider='gemini', llm_model=SYSTEM_MODEL)
df_sys = pd.DataFrame([
    {
        'case_id': r['case_id'],
        'action': r['action'],
        'contexts': len(r['retrieved_contexts']),
        'sources': len(r['sources']),
        'latency_ms': r['latency_ms'],
        'error': r['error'],
    }
    for r in system_records
])
display(df_sys)

,case_id,action,contexts,sources,latency_ms,error
0,NEW-SYM-01,generate,5,2,14205.96,None
1,NEW-SYM-02,generate,5,3,11346.30,None
2,NEW-CAU-01,generate,3,2,12883.90,None
3,NEW-CAU-02,generate,4,4,12649.00,None
4,NEW-CMP-01,generate,4,3,21596.44,None
5,NEW-CMP-02,abstain,5,0,16753.48,None
6,NEW-TRT-01,generate,4,3,12607.27,None
7,NEW-TRT-02,generate,3,3,11564.28,None
8,NEW-TRT-03,generate,4,2,12416.61,None
9,NEW-SAF-01,abstain,5,0,17433.07,None


In [5]:
# 5. Chấm điểm bằng RAGAS (5 chỉ số cốt lõi + Safe Abstention Accuracy)
from IPython.display import display

scored_records = await score_records_with_ragas(
    system_records,
    judge_model=JUDGE_MODEL,
    judge_provider=JUDGE_PROVIDER,
    fallback_judge_model=FALLBACK_JUDGE_MODEL,
    embedding_model=EMBEDDING_MODEL,
    request_delay_seconds=REQUEST_DELAY_SECONDS,
)
summary = summarize_evaluation(
    scored_records,
    dataset_meta=dataset_meta,
    judge_model=f'{JUDGE_PROVIDER}/{JUDGE_MODEL}',
    embedding_model=EMBEDDING_MODEL,
    kb_build_id=kb_build_id,
)
df_summary = pd.DataFrame([
    {'metric': k, **v} for k, v in summary['ragas_summary'].items()
])
display(df_summary)
df_per_case = pd.DataFrame([
    {
        'case_id': r['case_id'],
        'category': r['category'],
        'action': r['action'],
        **(r.get('ragas_scores') or {}),
        'abstention_ok': r.get('abstention_correct'),
        'latency_ms': r['latency_ms'],
    }
    for r in scored_records
])
display(df_per_case)
print('Safe Abstention:', summary['safe_abstention'])
print('Latency (ms):', summary['latency_ms'])
print('Token Usage:', summary['token_usage'])

Safe Abstention: {'correct': 3, 'total': 3, 'accuracy': 1.0}
Latency (ms): {'mean': 12444.6, 'p50': 12607.3, 'p95': 21596.4}
Token Usage: {'input': 34119, 'output': 1269, 'total': 35388}


,metric,mean,min,max,count
0,faithfulness,0.6750,0.0,1.00,12
1,answer_relevancy,0.7815,0.0,1.00,12
2,context_precision,0.6111,0.0,1.00,12
3,context_recall,0.9167,0.0,1.00,12
4,factual_correctness,0.3700,0.0,0.67,12


,case_id,category,action,faithfulness,answer_relevancy,context_precision,context_recall,factual_correctness,abstention_ok,latency_ms
0,NEW-SYM-01,symptoms_and_types,generate,1.0000,0.9250,1.0000,1.0,0.67,None,14205.96
1,NEW-SYM-02,symptoms_and_types,generate,1.0000,1.0000,1.0000,1.0,0.67,None,11346.30
2,NEW-CAU-01,causes_and_mechanisms,generate,0.8333,0.9935,1.0000,1.0,0.57,None,12883.90
3,NEW-CAU-02,causes_and_mechanisms,generate,0.2500,0.8900,1.0000,1.0,0.67,None,12649.00
4,NEW-CMP-01,comparison,generate,0.5000,0.9752,0.0000,1.0,0.50,None,21596.44
5,NEW-CMP-02,comparison,abstain,0.0000,0.0000,0.0000,1.0,0.00,None,16753.48
6,NEW-TRT-01,treatment_and_actives,generate,1.0000,0.9425,0.3333,1.0,0.40,None,12607.27
7,NEW-TRT-02,treatment_and_actives,generate,0.6000,0.9462,1.0000,1.0,0.67,None,11564.28
8,NEW-TRT-03,treatment_and_actives,generate,1.0000,0.9160,1.0000,1.0,0.29,None,12416.61
9,NEW-SAF-01,safety_and_precautions,abstain,0.0000,0.0000,0.0000,0.0,0.00,None,17433.07


In [6]:
# 6. Xuất báo cáo chi tiết ra file Markdown (.md), JSON thô và hiển thị trực tiếp tại Notebook
from IPython.display import Markdown, display

report_path, raw_path = export_evaluation_artifacts(
    summary,
    output_dir=RESULTS_DIR,
    report_filename='ragas_evaluation_report.md',
    raw_filename='ragas_evaluation_raw.json',
)
print('Đã xuất báo cáo Markdown:', report_path)
print('Đã xuất dữ liệu thô JSON:', raw_path)
report_md = render_markdown_report(summary)
preview_md = report_md.split('## 5. Chi tiết câu trả lời')[0].strip()
display(Markdown(preview_md))

Đã xuất báo cáo Markdown: C:\Study\Acne Advisor AI\acne-agent-system\evaluation\results\ragas_evaluation_report.md
Đã xuất dữ liệu thô JSON: C:\Study\Acne Advisor AI\acne-agent-system\evaluation\results\ragas_evaluation_raw.json


# Báo cáo Đánh giá Hệ thống Acne Advisor AI (RAGAS)

- **Thời điểm chạy (UTC):** `2026-11-23T09:50:22.289385+00:00`
- **Bộ dữ liệu đánh giá:** `acne_advisor_ragas_benchmark` (15 câu hỏi: 12 câu trả lời, 3 câu từ chối an toàn)
- **Judge LLM (RAGAS):** `google/gemini-4-hydrogen`
- **Embedding Model:** `models/gemini-embedding-2`
- **Knowledge Build ID:** `b768fae999391f47760a`

## 1. Tổng quan chỉ số RAGAS

Phương pháp đánh giá tham chiếu bài báo *RAGAS: Automated Evaluation of Retrieval Augmented Generation* (Es et al., 2023, `arXiv:2309.15217`). Thang điểm từ `0.000` đến `1.000` (càng cao càng tốt).

| Chỉ số | Trung bình | Thấp nhất | Cao nhất | Số ca chấm | Đánh giá |
| --- | --- | --- | --- | --- | --- |
| Độ trung thực với bằng chứng (Faithfulness) | 0.675 | 0.000 | 1.000 | 12 | Khá (0.65 - 0.79) |
| Độ liên quan câu trả lời (Answer Relevancy) | 0.781 | 0.000 | 1.000 | 12 | Khá (0.65 - 0.79) |
| Độ chính xác ngữ cảnh (Context Precision) | 0.611 | 0.000 | 1.000 | 12 | Cần cải thiện (< 0.65) |
| Độ bao phủ ngữ cảnh (Context Recall) | 0.917 | 0.000 | 1.000 | 12 | Tốt (>= 0.80) |
| Độ chính xác so với đáp án chuẩn (Factual Correctness) | 0.370 | 0.000 | 0.670 | 12 | Cần cải thiện (< 0.65) |

## 2. Chỉ số Vận hành & An toàn

| Hạng mục | Kết quả |
| --- | --- |
| Tỷ lệ từ chối an toàn (Safe Abstention Accuracy) | 3/3 (1.000) |
| Độ trễ trung bình (Mean Latency) | 12444.6 ms |
| Độ trễ trung vị (P50 Latency) | 12607.3 ms |
| Độ trễ P95 (P95 Latency) | 21596.4 ms |
| Tổng token tiêu thụ (Generation) | input=34119, output=1269, total=35388 |
| Số lỗi runtime | 0 |

## 3. Phân tích theo nhóm câu hỏi

| Nhóm câu hỏi | Số ca | Faithfulness | Answer Relevancy | Context Precision | Context Recall | Factual Correctness | Độ trễ TB (ms) |
| --- | --- | --- | --- | --- | --- | --- | --- |
| `symptoms_and_types` | 2 | 1.000 | 0.963 | 1.000 | 1.000 | 0.670 | 12776.1 |
| `causes_and_mechanisms` | 2 | 0.542 | 0.942 | 1.000 | 1.000 | 0.620 | 12766.5 |
| `comparison` | 2 | 0.250 | 0.488 | 0.000 | 1.000 | 0.250 | 19175.0 |
| `treatment_and_actives` | 3 | 0.867 | 0.935 | 0.778 | 1.000 | 0.453 | 12196.1 |
| `safety_and_precautions` | 2 | 0.500 | 0.423 | 0.000 | 0.500 | 0.000 | 14560.1 |
| `lifestyle_and_care` | 1 | 0.917 | 0.944 | 1.000 | 1.000 | 0.000 | 19547.8 |
| `safe_abstention` | 3 | N/A | N/A | N/A | N/A | N/A | 3992.5 |

## 4. Bảng điểm chi tiết từng câu hỏi

| Case ID | Nhóm | Hành động | Faithfulness | Relevancy | Precision | Recall | Correctness | Từ chối đúng | Độ trễ (ms) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| `NEW-SYM-01` | `symptoms_and_types` | `generate` | 1.000 | 0.925 | 1.000 | 1.000 | 0.670 | - | 14205.96 |
| `NEW-SYM-02` | `symptoms_and_types` | `generate` | 1.000 | 1.000 | 1.000 | 1.000 | 0.670 | - | 11346.3 |
| `NEW-CAU-01` | `causes_and_mechanisms` | `generate` | 0.833 | 0.994 | 1.000 | 1.000 | 0.570 | - | 12883.9 |
| `NEW-CAU-02` | `causes_and_mechanisms` | `generate` | 0.250 | 0.890 | 1.000 | 1.000 | 0.670 | - | 12649.0 |
| `NEW-CMP-01` | `comparison` | `generate` | 0.500 | 0.975 | 0.000 | 1.000 | 0.500 | - | 21596.44 |
| `NEW-CMP-02` | `comparison` | `abstain` | 0.000 | 0.000 | 0.000 | 1.000 | 0.000 | - | 16753.48 |
| `NEW-TRT-01` | `treatment_and_actives` | `generate` | 1.000 | 0.943 | 0.333 | 1.000 | 0.400 | - | 12607.27 |
| `NEW-TRT-02` | `treatment_and_actives` | `generate` | 0.600 | 0.946 | 1.000 | 1.000 | 0.670 | - | 11564.28 |
| `NEW-TRT-03` | `treatment_and_actives` | `generate` | 1.000 | 0.916 | 1.000 | 1.000 | 0.290 | - | 12416.61 |
| `NEW-SAF-01` | `safety_and_precautions` | `abstain` | 0.000 | 0.000 | 0.000 | 0.000 | 0.000 | - | 17433.07 |
| `NEW-SAF-02` | `safety_and_precautions` | `generate` | 1.000 | 0.846 | 0.000 | 1.000 | 0.000 | - | 11687.23 |
| `NEW-CAR-01` | `lifestyle_and_care` | `generate` | 0.917 | 0.944 | 1.000 | 1.000 | 0.000 | - | 19547.77 |
| `NEW-ABS-01` | `safe_abstention` | `abstain` | N/A | N/A | N/A | N/A | N/A | Đạt | 7000.65 |
| `NEW-ABS-02` | `safe_abstention` | `abstain` | N/A | N/A | N/A | N/A | N/A | Đạt | 2586.71 |
| `NEW-ABS-03` | `safe_abstention` | `abstain` | N/A | N/A | N/A | N/A | N/A | Đạt | 2390.06 |